# Einops Pack & Unpack: Deliberate Practice

This notebook teaches `pack` and `unpack` through worked examples followed by exercises (MathAcademy-style deliberate practice).

**Pattern**: Worked example → Exercise → Exercise → Exercise

In [ ]:
import torch
import numpy as np
from einops import rearrange, reduce, repeat, pack, unpack

---
## Section 1: Pack Basics — Combining Tensors

`pack` concatenates tensors along a wildcard `*` axis. The `*` absorbs "the rest" of each tensor's dims beyond the named axes. The return value `ps` (packed shapes) records what each input contributed so `unpack` can reverse it.

### Worked Example

Pack an RGB image with a depth map:

In [ ]:
rgb = torch.randn(64, 64, 3)    # h, w, c
depth = torch.randn(64, 64)      # h, w (no channel dim)
rgbd, ps = pack([rgb, depth], 'h w *')
print(rgbd.shape)   # (64, 64, 4) — the * absorbed 3 channels + 1 scalar = 4
print(ps)            # [(3,), ()] — records what each input contributed to *

**How it works:**
- `rgb` has shape `(64, 64, 3)`. The pattern `'h w *'` matches `h=64, w=64`, and `*` absorbs the remaining dim `(3,)`.
- `depth` has shape `(64, 64)`. The pattern matches `h=64, w=64`, and `*` absorbs nothing `()` — it's a scalar contribution.
- Pack concatenates along `*`: 3 + 1 = 4, giving shape `(64, 64, 4)`.
- `ps = [(3,), ()]` records that the first input contributed 3 and the second contributed a scalar.

### Exercise 1.1

Pack two feature maps of shape `(8, 32, 32)` and `(16, 32, 32)` along the channel dimension. Use pattern `'* h w'`. What's the output shape?

In [ ]:
feat1 = torch.randn(8, 32, 32)
feat2 = torch.randn(16, 32, 32)

# YOUR CODE HERE
# packed, ps = pack(...)
# print(packed.shape)  # expected: (24, 32, 32)
# print(ps)            # expected: [(8,), (16,)]

<details><summary>Solution</summary>

```python
packed, ps = pack([feat1, feat2], '* h w')
print(packed.shape)  # (24, 32, 32)
print(ps)            # [(8,), (16,)]
```

`*` absorbs the channel dim: 8 + 16 = 24.
</details>

### Exercise 1.2

Pack three tensors: a `(10, 64)` tensor, a `(10, 128)` tensor, and a `(10,)` tensor along the feature dimension. Pattern: `'batch *'`. What's the output shape and ps?

In [ ]:
a = torch.randn(10, 64)
b = torch.randn(10, 128)
c = torch.randn(10)

# YOUR CODE HERE
# packed, ps = pack(...)
# print(packed.shape)  # expected: (10, 193)
# print(ps)            # expected: [(64,), (128,), ()]

<details><summary>Solution</summary>

```python
packed, ps = pack([a, b, c], 'batch *')
print(packed.shape)  # (10, 193)
print(ps)            # [(64,), (128,), ()]
```

`*` absorbs feature dims: 64 + 128 + 1 (scalar) = 193.
</details>

---
## Section 2: Unpack Basics — Splitting Tensors

`unpack` reverses `pack`. It uses `ps` (packed shapes) to know how to split the wildcard axis back into the original shapes.

### Worked Example

Reverse the pack from Section 1:

In [ ]:
# Recreate the packed tensor
rgb = torch.randn(64, 64, 3)
depth = torch.randn(64, 64)
rgbd, ps = pack([rgb, depth], 'h w *')

# Unpack
rgb_recovered, depth_recovered = unpack(rgbd, ps, 'h w *')
print(rgb_recovered.shape)    # (64, 64, 3)
print(depth_recovered.shape)  # (64, 64)

**How it works:** Unpack uses `ps` to know exactly how to split the wildcard axis back into the original shapes. The scalar (depth) with `ps=()` gets its trailing dim removed automatically.

### Exercise 2.1

Pack `a = torch.randn(4, 10, 32)` and `b = torch.randn(4, 10, 64)` with pattern `'b t *'`, then unpack them. Verify shapes match originals.

In [ ]:
a = torch.randn(4, 10, 32)
b = torch.randn(4, 10, 64)

# YOUR CODE HERE
# Step 1: Pack
# packed, ps = pack(...)
# print(packed.shape)  # expected: (4, 10, 96)

# Step 2: Unpack
# a_rec, b_rec = unpack(...)
# print(a_rec.shape)   # expected: (4, 10, 32)
# print(b_rec.shape)   # expected: (4, 10, 64)

# Step 3: Verify
# assert torch.allclose(a, a_rec)
# assert torch.allclose(b, b_rec)

<details><summary>Solution</summary>

```python
packed, ps = pack([a, b], 'b t *')
print(packed.shape)  # (4, 10, 96)

a_rec, b_rec = unpack(packed, ps, 'b t *')
print(a_rec.shape)   # (4, 10, 32)
print(b_rec.shape)   # (4, 10, 64)

assert torch.allclose(a, a_rec)
assert torch.allclose(b, b_rec)
```
</details>

### Exercise 2.2

Given `packed = torch.randn(4, 10, 96)`, manually unpack into two tensors of feature dims 32 and 64 **without** using `ps` from a previous pack. Hint: pass `[[32], [64]]` as the shapes argument.

In [ ]:
packed = torch.randn(4, 10, 96)

# YOUR CODE HERE
# x, y = unpack(packed, ..., 'b t *')
# print(x.shape)  # expected: (4, 10, 32)
# print(y.shape)  # expected: (4, 10, 64)

<details><summary>Solution</summary>

```python
x, y = unpack(packed, [[32], [64]], 'b t *')
print(x.shape)  # (4, 10, 32)
print(y.shape)  # (4, 10, 64)
```

You can manually specify the shapes instead of relying on `ps` from a previous pack call. Each inner list describes what that output's wildcard dims should be.
</details>

---
## Section 3: Auto-Batching with Pack/Unpack

A powerful pattern: write functions that work on both single items AND batches by using `*` to absorb (or not absorb) a batch dimension.

### Worked Example

A function that works on both single images and batches:

In [ ]:
def process(x):
    # x could be (h, w, c) or (b, h, w, c)
    x, ps = pack([x], '* h w c')         # always becomes (b, h, w, c) with b=1 if needed
    # ... do processing on batched tensor ...
    result = x * 2  # dummy processing
    [result] = unpack(result, ps, '* h w c')  # restore original shape
    return result

single = torch.randn(64, 64, 3)
batched = torch.randn(8, 64, 64, 3)
print(process(single).shape)    # (64, 64, 3) — no batch dim
print(process(batched).shape)   # (8, 64, 64, 3) — batch preserved

**How it works:**
- When you pack a **single tensor** in a list, `*` absorbs all leading dims not accounted for by the named axes.
- For a single image `(h, w, c)` there are no leading dims, so `*` is empty and a dim of 1 is added → `(1, h, w, c)`.
- For a batch `(b, h, w, c)`, `*` absorbs `b` → shape stays `(b, h, w, c)`.
- Unpack restores the original shape: removes the added dim for singles, keeps it for batches.

### Exercise 3.1

Write a `normalize` function that subtracts the mean and divides by std along the last dim. It should work on both `(seq_len, features)` and `(batch, seq_len, features)` inputs. Use pack/unpack with pattern `'* features'`.

In [ ]:
def normalize(x):
    # YOUR CODE HERE
    # 1. Pack x with pattern '* features' to ensure a leading batch dim
    # 2. Subtract mean and divide by std along the last dim (dim=-1, keepdim=True)
    # 3. Unpack to restore original shape
    pass

# Test:
# unbatched = torch.randn(20, 64)
# batched = torch.randn(4, 20, 64)
# print(normalize(unbatched).shape)  # expected: (20, 64)
# print(normalize(batched).shape)    # expected: (4, 20, 64)

<details><summary>Solution</summary>

```python
def normalize(x):
    x, ps = pack([x], '* features')
    x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-8)
    [x] = unpack(x, ps, '* features')
    return x

unbatched = torch.randn(20, 64)
batched = torch.randn(4, 20, 64)
print(normalize(unbatched).shape)  # (20, 64)
print(normalize(batched).shape)    # (4, 20, 64)
```
</details>

### Exercise 3.2

Write a `channel_swap` function that reverses the channel order of an image. Should work on `(c, h, w)` and `(b, c, h, w)`. Use pack/unpack.

In [ ]:
def channel_swap(x):
    # YOUR CODE HERE
    # 1. Pack with pattern '* c h w'
    # 2. Reverse channel dim: x = x[:, torch.arange(x.shape[1]-1, -1, -1)]
    #    or simply x = x.flip(dims=[1])
    # 3. Unpack to restore original shape
    pass

# Test:
# single_img = torch.randn(3, 32, 32)
# batch_img = torch.randn(8, 3, 32, 32)
# print(channel_swap(single_img).shape)  # expected: (3, 32, 32)
# print(channel_swap(batch_img).shape)   # expected: (8, 3, 32, 32)

<details><summary>Solution</summary>

```python
def channel_swap(x):
    x, ps = pack([x], '* c h w')
    x = x.flip(dims=[1])
    [x] = unpack(x, ps, '* c h w')
    return x

single_img = torch.randn(3, 32, 32)
batch_img = torch.randn(8, 3, 32, 32)
print(channel_swap(single_img).shape)  # (3, 32, 32)
print(channel_swap(batch_img).shape)   # (8, 3, 32, 32)
```
</details>

---
## Section 4: Vision Transformer Class Token

A common pattern in ViTs: prepend special tokens to a sequence, process everything together, then separate them back out.

### Worked Example

Prepend a class token to patch tokens, process, then separate:

In [ ]:
batch_size = 4
n_patches = 196
embed_dim = 768

class_tokens = torch.randn(batch_size, 1, embed_dim)     # (b, 1, c)
patch_tokens = torch.randn(batch_size, n_patches, embed_dim)  # (b, 196, c)

# Pack: concatenate along token dimension
tokens, ps = pack([class_tokens, patch_tokens], 'b * c')
print(tokens.shape)  # (4, 197, 768)

# ... transformer processing ...
processed = tokens  # (would be transformer output)

# Unpack: separate class token from patch tokens
class_out, patches_out = unpack(processed, ps, 'b * c')
print(class_out.shape)    # (4, 1, 768)
print(patches_out.shape)  # (4, 196, 768)

**How it works:** The `*` absorbs the token dimension. Class token contributes 1, patches contribute 196. Pack concatenates to 197. Unpack splits back using `ps`. This replaces manual `torch.cat` + indexing.

### Exercise 4.1

Implement the same pattern but with **two** special tokens — a `[CLS]` and a `[SEP]` token, each shape `(b, 1, c)` — prepended to a sequence of `(b, 50, c)` text tokens. Pack all three, then unpack.

In [ ]:
b, c = 4, 256
cls_token = torch.randn(b, 1, c)
sep_token = torch.randn(b, 1, c)
text_tokens = torch.randn(b, 50, c)

# YOUR CODE HERE
# Step 1: Pack all three with pattern 'b * c'
# tokens, ps = pack(...)
# print(tokens.shape)  # expected: (4, 52, 256)

# Step 2: Unpack back
# cls_out, sep_out, text_out = unpack(...)
# print(cls_out.shape)   # expected: (4, 1, 256)
# print(sep_out.shape)   # expected: (4, 1, 256)
# print(text_out.shape)  # expected: (4, 50, 256)

<details><summary>Solution</summary>

```python
tokens, ps = pack([cls_token, sep_token, text_tokens], 'b * c')
print(tokens.shape)  # (4, 52, 256)

cls_out, sep_out, text_out = unpack(tokens, ps, 'b * c')
print(cls_out.shape)   # (4, 1, 256)
print(sep_out.shape)   # (4, 1, 256)
print(text_out.shape)  # (4, 50, 256)
```
</details>

### Exercise 4.2

Pack three modalities: text tokens `(b, 20, 256)`, image tokens `(b, 49, 256)`, and audio tokens `(b, 100, 256)`. Process together, then unpack. What's the packed shape?

In [ ]:
b, c = 2, 256
text = torch.randn(b, 20, c)
image = torch.randn(b, 49, c)
audio = torch.randn(b, 100, c)

# YOUR CODE HERE
# Step 1: Pack with pattern 'b * c'
# multimodal, ps = pack(...)
# print(multimodal.shape)  # expected: (2, 169, 256)

# Step 2: Unpack
# text_out, image_out, audio_out = unpack(...)
# print(text_out.shape)   # expected: (2, 20, 256)
# print(image_out.shape)  # expected: (2, 49, 256)
# print(audio_out.shape)  # expected: (2, 100, 256)

<details><summary>Solution</summary>

```python
multimodal, ps = pack([text, image, audio], 'b * c')
print(multimodal.shape)  # (2, 169, 256)

text_out, image_out, audio_out = unpack(multimodal, ps, 'b * c')
print(text_out.shape)   # (2, 20, 256)
print(image_out.shape)  # (2, 49, 256)
print(audio_out.shape)  # (2, 100, 256)
```

Packed shape is `(2, 169, 256)` because 20 + 49 + 100 = 169 tokens.
</details>

---
## Section 5: Multi-Output Predictions with Pack/Unpack

Use `unpack` with manually specified shapes to split a tensor into semantically different parts — no prior `pack` needed.

### Worked Example

A detection head that predicts multiple things per spatial location:

In [ ]:
# Model outputs (b, h, w, 8) — but 8 encodes multiple things:
# - confidence (1 value)
# - bbox deltas (4 values)
# - class logits (3 values)
output = torch.randn(4, 16, 16, 8)

confidence, bbox, classes = unpack(
    output,
    [[], [4], [3]],  # [] = scalar, [4] = 4-vector, [3] = 3-vector
    'b h w *'
)
print(confidence.shape)  # (4, 16, 16) — scalar, no trailing dim
print(bbox.shape)         # (4, 16, 16, 4)
print(classes.shape)      # (4, 16, 16, 3)

**How it works:** You can unpack with manually specified shapes instead of `ps`. `[]` means the contribution is a scalar (trailing dim removed). `[n]` means n values along the wildcard axis.

### Exercise 5.1

A model outputs `(b, t, 7)` for each timestep. Unpack into: position `(3 values)`, velocity `(3 values)`, and done_flag `(scalar)`. Use pattern `'b t *'`.

In [ ]:
model_output = torch.randn(2, 50, 7)

# YOUR CODE HERE
# position, velocity, done_flag = unpack(...)
# print(position.shape)   # expected: (2, 50, 3)
# print(velocity.shape)   # expected: (2, 50, 3)
# print(done_flag.shape)  # expected: (2, 50)

<details><summary>Solution</summary>

```python
position, velocity, done_flag = unpack(model_output, [[3], [3], []], 'b t *')
print(position.shape)   # (2, 50, 3)
print(velocity.shape)   # (2, 50, 3)
print(done_flag.shape)  # (2, 50)
```

`[]` makes `done_flag` a scalar (no trailing dim). `[3]` gives 3 values along `*`.
</details>

### Exercise 5.2

Pack the reverse: given `position = torch.randn(2, 50, 3)`, `velocity = torch.randn(2, 50, 3)`, and `done = torch.randn(2, 50)`, pack them into a single tensor along the last dim. Verify shape is `(2, 50, 7)`.

In [ ]:
position = torch.randn(2, 50, 3)
velocity = torch.randn(2, 50, 3)
done = torch.randn(2, 50)

# YOUR CODE HERE
# packed, ps = pack(...)
# print(packed.shape)  # expected: (2, 50, 7)
# print(ps)            # expected: [(3,), (3,), ()]

<details><summary>Solution</summary>

```python
packed, ps = pack([position, velocity, done], 'b t *')
print(packed.shape)  # (2, 50, 7)
print(ps)            # [(3,), (3,), ()]
```

3 + 3 + 1 (scalar) = 7 along the `*` axis.
</details>

---
## Section 6: Challenge

Put it all together.

### Exercise 6.1

Build a complete mini-pipeline:
1. Start with `images = torch.randn(4, 3, 32, 32)` and `labels = torch.randn(4, 10)`
2. Pack images and labels together using pattern `'b *'` (flatten everything per sample)
3. Print the packed shape and `ps`
4. Unpack back and verify shapes match originals

In [ ]:
images = torch.randn(4, 3, 32, 32)
labels = torch.randn(4, 10)

# YOUR CODE HERE
# Step 1: Pack with pattern 'b *'
# packed, ps = pack(...)
# print(packed.shape)  # expected: (4, 3082) — 3*32*32 + 10 = 3072 + 10 = 3082
# print(ps)            # expected: [(3, 32, 32), (10,)]

# Step 2: Unpack
# images_rec, labels_rec = unpack(...)
# print(images_rec.shape)  # expected: (4, 3, 32, 32)
# print(labels_rec.shape)  # expected: (4, 10)

# Step 3: Verify
# assert torch.allclose(images, images_rec)
# assert torch.allclose(labels, labels_rec)
# print("All checks passed!")

<details><summary>Solution</summary>

```python
images = torch.randn(4, 3, 32, 32)
labels = torch.randn(4, 10)

# Step 1: Pack
packed, ps = pack([images, labels], 'b *')
print(packed.shape)  # (4, 3082) — 3*32*32 + 10 = 3082
print(ps)            # [(3, 32, 32), (10,)]

# Step 2: Unpack
images_rec, labels_rec = unpack(packed, ps, 'b *')
print(images_rec.shape)  # (4, 3, 32, 32)
print(labels_rec.shape)  # (4, 10)

# Step 3: Verify
assert torch.allclose(images, images_rec)
assert torch.allclose(labels, labels_rec)
print("All checks passed!")
```

Note how `*` flattens multi-dimensional contributions: `(3, 32, 32)` becomes 3072 values, and `(10,)` becomes 10 values. Total: 3082. Unpack restores the original multi-dimensional shapes using `ps`.
</details>